[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/developability/developability.ipynb)

# Developable Surfaces

This example flows a curved surface, the Stanford bunny, towards a developable one, following Stein, Grinspun and Crane. A developable surface can be unrolled flat without stretching or tearing, like a sheet of paper bent into a cylinder or a cone; it bends in one direction only at each point. The bunny's rounded flanks bend both ways almost everywhere, and flowing it towards developability creases it into patches that could each be cut from a flat sheet, joined along sharp folds.

How far a vertex is from developable is the smallest eigenvalue of a map on directions, the sum of the dyads of the face normals around it, `normal * (normal | Vector)`: zero when one direction meets every normal at a right angle. The vertices and edges are points, antivectors, intrinsic like all geometry, so two edges meet in their face's normal, `edge & edge`, twice its area long. The energy's gradient is a vector, the partner a vertex's step pairs with, `gradient & step`, the energy's change; each face's share of it is the inner product of a vector along the face with an edge, `across | edge`, and the metric, `.dual()`, turns the gradient into a step.

In the notation of differential geometry a developable surface reads as one of zero Gaussian curvature, $K = \kappa_1 \kappa_2 = 0$, whose Gauss map sends every neighbourhood onto a curve of the unit sphere.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import stack

from numga.algebras import VGA3D as ga
from examples.animation import show_animation
from examples.surfaces.developability import render
from examples.surfaces.developability.scenarios import bunny
from examples.surfaces.developability.core import Mesh, as_scalar, mv

In [ ]:
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Point = ga.gatype.antivector()

## 1. The spread of the normals around a vertex

A smooth surface is developable where its normals, all around a point, lie on one great circle of the sphere of directions: the surface bends about one direction only, the direction of its rulings, the straight lines that run through a developable surface, and the normals turn about that direction. At a vertex of a triangle mesh, the normals of the faces around it lie on one great circle exactly when one direction meets every one of them at a right angle. The spread of the normals, the sum of their dyads `normal * (normal | Vector)` over the faces around the vertex, each weighted by the face's angle at the vertex, is a map whose quadratic form measures, for a direction `u`, how much the normals lean along it, `sum(angle * (u | normal)**2)`. Its smallest eigenvalue is zero when some direction is perpendicular to every normal, and grows as the normals spread out over the sphere. The developability energy is the sum of the smallest eigenvalues over the vertices.

The angles make the spread a property of the surface rather than of its triangles: a face split in two keeps its angle at the vertex, so splitting faces changes nothing. Without them, a flow can lower the energy by squeezing faces into slivers instead of creasing the surface.

In matrix notation the energy reads $E = \sum_i \lambda_{\min}\left(\sum_{ijk} \theta_i^{jk} N_{ijk} N_{ijk}^{\mathsf T}\right)$, with $\theta_i^{jk}$ the angle of face $ijk$ at vertex $i$.

In [ ]:
def spread(mesh: Mesh) -> tuple[Vector, Vector]:
    """The unit normal of each face, and the spread of the face normals around each vertex, the sum of
    their dyads, each weighted by the face's angle at the vertex, a map `Vector <- Vector` at each vertex."""
    edges = mesh.triangle_edges                                                 # [3] Point[F]
    # two edges meet in the face's normal
    normals = (edges[0] & edges[1]).normalized()                                # Vector[F]
    # Each face to the vertices at its corners through its angles there, run back: each vertex sums over
    # its faces.
    around = ~mesh.at_corners(mesh.corner_cosines().arccos())                   # [V, F] Scalar
    return normals, around * (normals * (normals | Vector))                    # Vector[F], [V] Vector <- Vector


rabbit = bunny()
normals, spreads = spread(rabbit)
values, vectors = spreads.batch().eigh()                                        # [V, 3] Scalar, [V, 3] Vector
print("the bunny's developability energy:", values[:, 0].sum(axis=0).to_array())

## 2. The gradient

The flow moves every vertex down the energy's gradient. The smallest eigenvalue is the least value of the spread's form over unit directions, so to first order its change is the change of the form at the least direction `u`, held fixed: each face's term, its angle times `(u | normal)**2`, changes as the face's normal turns and as its angle opens. Each face's shares, run back to its corners, add up at the vertices, and the gradient is a vector: paired with a vertex's step, `gradient & step`, it gives the energy's change.

In [ ]:
def gradient(mesh: Mesh, normals: Vector, directions: Vector) -> Vector:
    """The energy's gradient at each vertex, with each vertex's least direction `[V] Vector` held fixed."""
    edges = mesh.triangle_edges.batch()                                         # [3, F] Point
    double_areas = (edges[0] & edges[1]).norm()                                 # [F] Scalar
    face_normals = normals.batch()                                              # [F] Vector
    angles = mesh.corner_cosines().batch().arccos()                             # [3, F] Scalar
    corner_directions = directions[mesh.faces.T]                                # [3, F] Vector
    along = corner_directions | face_normals                                    # [3, F] Scalar
    # As the normal turns: the angle-weighted dyads of the corners' least directions, applied to the face's
    # normal, the inner product of their part along the face with the edge facing each corner, over
    # twice the area.
    pulled = (corner_directions * (along * angles)).sum(axis=0)                 # [F] Vector
    across = 2 * (pulled - face_normals * (face_normals | pulled))             # [F] Vector
    turning = -(across | edges) / double_areas                                  # [3, F] Vector
    # As the angle opens: the edges from each corner to the next and to the one after, and their join.
    ahead, behind = edges[[2, 0, 1]], -edges[[1, 2, 0]]                         # [3, F] Point each
    corner = ahead & behind                                                     # [3, F] Vector
    weight = along * along / corner.norm()                                      # [3, F] Scalar
    # An edge's inner product with the corner turns the edge a quarter turn within the face.
    to_next = -(ahead | corner) / ahead.scalar_norm_squared() * weight          # [3, F] Vector
    to_after = (behind | corner) / behind.scalar_norm_squared() * weight        # [3, F] Vector
    opening = to_next[[2, 0, 1]] + to_after[[1, 2, 0]] - to_next - to_after     # [3, F] Vector
    return ~mesh.at_corners(turning + opening) * as_scalar(np.ones(len(mesh.faces))).field()   # Vector[V]


print("largest gradient on the bunny:", gradient(rabbit, normals, vectors[:, 0]).norm().to_array().max())

## 3. The flow

Each step of the flow computes the spread and its least directions afresh, and moves every vertex a small step down the gradient, divided by how sharply the energy curves at the vertex. Moving a corner turns its face's normal by the step over the corner's height above the opposite edge, so each face adds its angle at the vertex over that height squared. A step of the same size everywhere overshoots where triangles are small or thin, at the tips of the ears first, and the flow jitters there; over the stiffness, every vertex moves by the same share of what its faces can take.

The flow cell takes about twenty seconds: six hundred steps of about thirty milliseconds, half of each spent on the gradient.

In [ ]:
# The flow's step, as a share of what each vertex's faces can take, its number of steps, the steps
# between kept states, and the duration of a frame.
step = 0.3
n_steps = 600
stride = 10
duration_ms = 80

In [ ]:
def stiffness(mesh: Mesh) -> Scalar:
    """How sharply the energy curves at each vertex: each face's angle at the vertex over the vertex's
    height above the opposite edge squared, the edge's length squared over `double_areas**2`."""
    edges = mesh.triangle_edges.batch()                                         # [3, F] Point
    double_areas = (edges[0] & edges[1]).norm()                                 # [F] Scalar
    angles = mesh.corner_cosines().batch().arccos()                             # [3, F] Scalar
    return ~mesh.at_corners(angles * edges.scalar_norm_squared() / double_areas.squared()) * as_scalar(np.ones(len(mesh.faces))).field()


def flow(mesh: Mesh, step: float) -> Iterator[tuple[Point, Scalar]]:
    """Gradient descent on the developability energy, each vertex's step the gradient over the stiffness:
    at every step, without end, the vertices and each vertex's smallest eigenvalue."""
    vertices = mesh.vertices
    while True:
        current = mesh.copy(vertices)
        normals, spreads = spread(current)
        values, vectors = spreads.batch().eigh()                                # [V, 3] Scalar, [V, 3] Vector
        yield vertices, values[:, 0]
        # The gradient pairs with a step; the metric turns it into one.
        vertices = vertices - step * (gradient(current, normals, vectors[:, 0]) / stiffness(current)).dual()


states = list(islice(flow(rabbit, step), 0, n_steps + 1, stride))
energies = stack([values.sum(axis=0) for _, values in states])                  # [states] Scalar

In [ ]:
print("energy before and after:")
print(energies[0].coefficient_table())
print(energies[-1].coefficient_table())

## 4. The bunny creasing

The flow, every tenth step, with every face lit flat by its own normal, so a developable patch reads as a smooth sweep of shade and a fold as a sharp change of it. The bumps of the scan smooth out first; then the rounded flanks, back and head flatten into a few large patches, and the folds between them sharpen. Along a fold the normals take two directions only, so a fold is developable too.

In [ ]:
show_animation(render.animate_flow(rabbit, states), "developability_flow", duration_ms)

In [ ]:
# checks
# Every kept state has less energy than the one before, and the flow takes away nearly all of it.
assert np.all(np.diff(energies) < 0)
assert energies[-1] < 0.1 * energies[0]